# run_all_indvA: the full pipeline for dataset A, individual questionnaire items

This notebook trains the five models on dataset A and computes every evaluation
table for this representation. The sections follow the order of the analysis: the data, the models,
classification performance, the two uncertainty thresholds, then four families of evaluation.

**Question.** At a matched review volume, do conformal prediction sets and an entropy threshold flag
the same patients? Does either review queue hold a larger share of the model's errors?

The notebook computes and saves results. Each execution writes to its own timestamped folder under
`results/`, with a manifest recording the configuration, the data file hashes and the package
versions. A smoke run carries the prefix `smoke_`, which keeps it apart from full runs.

**Representation.** 86 features. Each questionnaire item is divided by its maximum possible score: PHQ and GAD items by 3, SIAS and PDSS by 4, FQ by 8.

**Evaluation order.** eval_2 builds the thresholds and runs first, since every later evaluation uses
them. eval_1, the validity check, runs straight after it.

## 0. Configuration

The settings live in `pipeline/config.py`. This cell sets the representation, the smoke switch and a
one-line run note, which goes into the manifest.

In [ ]:
# autoreload picks up an edit to a .py file without restarting the kernel, which keeps
# the trained models in memory.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from pipeline import config as C
from pipeline import data_io, checks
from models import run_models
from pipeline import report as R
from evaluation import stats as S
from evaluation import eval_0_performance as eval_0
from evaluation import eval_1_validity    as eval_1
from evaluation import eval_2_flags       as eval_2
from evaluation import eval_3_concordance as eval_3
from evaluation import eval_4_error       as eval_4
from evaluation import eval_5_sensitivity as eval_5

# REPRESENTATION selects the dataset this notebook runs.
REPRESENTATION = "indvA"

# SMOKE = True runs reduced draws, grids and epochs, which tests the whole pipeline in
# minutes. Smoke runs are for testing only.
C.SMOKE = False

RUN_NOTE = "full run"


print(f"Representation: {C.REPRESENTATIONS[REPRESENTATION]['label']}")
print(f"M samples:      {C.M_SAMPLES}")
print(f"alpha:          {C.CP_ALPHA}   sweep {C.ALPHA_GRID}")
print(f"Review rates:   {C.REVIEW_RATES}")
print(f"Scores:         {C.CONFORMAL_SCORES} (primary: {C.PRIMARY_SCORE})")
print(f"Bootstrap:      {C.N_BOOT} resamples at {int(C.CI_LEVEL*100)}%")

### 0b. Environment

A quick check, before any data loads, that the kernel imports every package the five models need. In
the environment built from `environment.yml`, all five models report as trainable. A blocked model
points to an incomplete environment, and rebuilding it from `environment.yml` is the fix.

In [ ]:
# env: the package report and the models that can train.
env = checks.check_environment()

In [ ]:
# run: the run context and its folder. The folder exists from this point, so a failed
# run still leaves a record.
run = data_io.new_run(REPRESENTATION, note=RUN_NOTE)
cfg = C.base_config(REPRESENTATION, run["run_dir"])

## 1. Data

The training, calibration and test splits load exactly as stored. The calibration split estimates the
conformal threshold, and the entropy threshold then follows from the flag rate this conformal
threshold produces on calibration. Pointing `config.REPRESENTATIONS` at other files runs the pipeline
on different data.

In [ ]:
# data: X and y per split, plus the patient indices.
data = data_io.load_splits(run)

In [ ]:
# The training features before any check runs, for comparison with the preflight report.
preview = data_io.preview(data)
display(preview.head(12))
print(f"... {len(preview)} features total")
print(f"\nGlobal range across the training split: "
      f"{preview['min'].min():.3f} to {preview['max'].max():.3f}")

## 2. Preflight gate

The gate checks for faults that make a model fail quietly: values outside 0 to 1, missing values,
non-numeric columns, constant columns, patients in two splits, labels outside 1 to 4, and a
calibration class too small for a class-level estimate. Each fault names the models it threatens.
Features outside their expected range threaten the bootstrap logistic regression, the SVM ensemble and
the dropout network, since their penalties and gradients assume features of comparable size. The tree
ensemble is scale invariant.

**The cell halts on any finding.** The resolution column states the change to make before the cell
runs again.

Two findings are notes only. A rare one-hot category can be absent from a split by chance. Two
income levels carry 8 and 12 positives across 2,525 patients, so a 16 percent calibration share
often holds none. These columns stay in, and a penalised model shrinks them towards zero. The second
note, a small excursion beyond 0 to 1 in calibration or test, applies only when a training-fit
scaler is switched on.

In [ ]:
# PreflightError on a finding is the gate working as designed.
pf = checks.run_preflight(run, data)

### 2b. Scaling

Features keep their instrument scale, and this cell records the scaling rule in the run manifest.
Both representations arrive on 0 to 1, since every questionnaire item or summed score is divided by
its maximum possible score during data preparation. The divisors are fixed properties of the
questionnaires and independent of the patients, so the exchangeability assumption behind the conformal
guarantee holds. The instrument scale also keeps the two representations comparable.

Setting `C.SCALE_ON_LOAD = True` in section 0 switches to a min-max scaler fit on the training split,
and the preflight gate adapts to either rule.

In [ ]:
data = data_io.scale_splits(run, data)

In [ ]:
# Second pass. A clean result clears the data for training.
pf = checks.run_preflight(run, data)
display(pf["risk"][["Model", "n_faults", "faults", "verdict"]])

## 3. Model training

One cell trains the five models, each producing M draws of the predictive distribution per patient on
all three splits. `MODELS_TO_RUN` holds the models the preflight gate cleared that this kernel can
import. Editing it drops a model or retrains a single one. A model already fitted in this run folder
is reloaded, so a kernel restart keeps every finished fit. Each model prints its outputs and the
results they feed.

In [ ]:
# MODELS_TO_RUN: the models the gate cleared that this kernel can import. Setting it
# to one key retrains that model alone.
MODELS_TO_RUN = [m for m in pf["safe_models"] if m in env["trainable"]]

print(f"About to train: {MODELS_TO_RUN}\n")
all_models = run_models.train_all(run, data, MODELS_TO_RUN, reuse=True)

## 4. Model performance

Discrimination and calibration for every model, computed in a separate pass after training. Each model
is reported as fitted, and ECE, Brier score and NLL measure the quality of its probabilities.

In [ ]:
perf = eval_0.run(run, all_models, data, cfg)

## 5. Uncertainty: the two thresholds

1. The conformal threshold q is the conformal quantile of the calibration nonconformity scores at the
   pre-specified alpha.
2. Prediction sets are built on calibration, and r is the share of calibration patients the conformal
   flag marks uncertain, with an empty set or a set of two or more classes. r is the review rate
   conformal prediction implies.
3. The entropy threshold t is the (1 - r) quantile of calibration entropy, so both methods flag the
   same share of calibration patients.
4. Both thresholds then apply to the test split.

The calibration and test flag rates print side by side. The rates are matched on calibration only,
so a gap on test is a result in its own right. A large gap would weaken the exchangeability
assumption behind the conformal guarantee.

In [ ]:
# uq: the per-model state every later evaluation uses.
uq = eval_2.run(run, all_models, data, cfg, alpha=C.CP_ALPHA)

## 6. Evaluations

Four families of evaluation, in the order the results report them.

### 6a. Conformal validity and set efficiency

The entropy threshold comes from the conformal flag rate, so the conformal sets are checked first.
The checks cover coverage on test against the 1 - alpha target, coverage within each treatment
class, and the share of flagged sets holding three or four classes. A set holding almost every class
leaves a reviewer little to choose between.

In [ ]:
validity = eval_1.run(run, all_models, uq, data, alpha=C.CP_ALPHA)

### 6b. Concordance

Agreement between the two flags comes from several angles. These are the cross of the two flags,
kappa beyond chance, PABAK with positive and negative agreement for prevalence, and agreement within
each treatment class. The last is the Spearman correlation between entropy and the second-largest
mean probability, the exact continuous conformal score. Every statistic carries a patient-level
bootstrap interval.

In [ ]:
concordance = eval_3.run(run, all_models, uq, data)

### 6c. Error detection

Error recall rises with the flag rate: a queue holding 70 percent of patients at random contains about
70 percent of the errors. Every recall value therefore sits beside the random queue of the same size,
and the difference carries an interval.

At alpha 0.05 both flag rates exceed a feasible review volume, so the comparison repeats at fixed
review budgets of 5, 10, 20, 30 and 40 percent. There both methods flag the same number of patients.

The combination rules test union and intersection at their own volume, and two rank rules cut to the
volume of the better single flag. The matched-volume comparison is the test of whether the two flags
add to each other.

In [ ]:
error = eval_4.run(run, all_models, uq, data)

### 6d. Sensitivity

Four analyses, one table each: the alpha sweep, resampling of the 404 calibration patients, APS in
place of LAC, and convergence in the number of draws M. Each analysis post-processes the stored
probability arrays, so the section runs in minutes.

In [ ]:
sensitivity = eval_5.run(run, all_models, uq, data)

## 7. Save and inventory

Each table is already in the run folder. This cell closes the manifest and prints the saved outputs,
grouped by evaluation family.

In [ ]:
inventory = R.finalise(run)

## 8. Reloading this run

A fresh kernel reloads the newest full run from its stored fits:

```python
run_dir    = data_io.find_run("indvA")                 # newest full run
all_models = run_models.load_all(run_dir, "indvA")
uq         = data_io.load_pickle(next(run_dir.glob("eval_2_*/uq_state_*.pkl")))
```